# Committor from an infretis StapleTIS Run (2D Z Potential, Underdamped Langevin)

Fits the committor of the **2D Z potential with Langevin dynamics** (γ = 2, kT = 0.05, m = 1, dt = 0.002) from the paths of an **infretis StapleTIS** simulation (ASE engine, `integrator = "langevin"`) with NPNEq (`CommittorNE`, CuPy). The committor depends on the velocities as well, q(y, x, v_y, v_x), so, as in `infretis_staple_1d_langevin.ipynb`, there are two fits that bring the velocity in differently: **positions (y, x) with histories**, and **positions and the recorded velocities (y, x, v_y, v_x) without histories**.

**How the paths become training data** (the same code as `infretis_staple_2d_z.ipynb`: `tis_data.prepare_data`, which switches to `infretis_data.py` for infretis runs; details in `docs/npneq_tis_notes.md`, sections 3, 9 and 15):
- **Read with infretis' own definitions:** interfaces and swap settings from `infretis.toml`, one row per finished path from `infretis_data.txt`, the order parameters from `load/<pnr>/order.txt` (or an h5 archive), and each path's core in an ensemble from infretis' `StaplePath.get_sh_region`.
- **Cut at M, mirrored pieces:** every (path, ensemble) pair gives the piece from its first frame past M; with `REVERSED` also the part before the last M crossing, read backwards. **Velocities are odd under time reversal:** the velocity columns (`VEL_COLUMNS`) are negated in the mirrored pieces (`--odd-columns`). Section 4 checks the sign: the recorded velocity must follow the finite difference of the positions in every piece.
- **Weights:** `path_f` (`'rejections'`) for committor pieces; `'staple'` = `path_f / path_w` is for the stored staple paths (rates).

**The run:** `loop_z_lgv/production_seed` of the learning loop (`docs/op_loop_z_lgv_workstation.md`): the Z potential (`sjoelbak_angle:RectangularGridWithAngledBarrierPotential(Lx = 0.3, angle = 45)`, a 45° ridge and a constant force along y) with ASE's Langevin integrator, seed order parameter λ = y, A: y ≤ 0.05, B: y ≥ 0.85, 11 interfaces 0.05, 0.13, …, 0.85, StapleTIS with infinite swap. **`order.txt` has the time, λ, y, x, v_y, v_x**, so after the time (column 0) the first column is a copy of the CV y: the cell after the data loading drops it and keeps (y, x, v_y, v_x). Any other infretis run with the same columns works (the learned-λ production also records them, but its λ is not y: then set `LAMBDA_A` and `LAMBDA_B` to its state boundaries in λ, and use y, x, v_y, v_x for the CVs).

**Reference:** there is no exact phase-space committor of this chain. `2d_zpot/exact/langevin_g2/qbar_maxwell.npz` has the **velocity-averaged** committor q̄(y, x) (Maxwell velocities, 10000 shots per point) of the toytis BBK chain at the same γ, kT, dt. Two caveats: it is not the same integrator as ASE's Langevin (a small difference), and it averages over v, so only the mean of a fit over the velocities of the frames in a (y, x) bin is comparable with it. The notebook therefore compares **bin means** on the (y, x) plane and along y; differences of a few 0.01 are not errors. For q at fixed velocities the fits are judged by the convergence metric `zqs_z` and by the velocity planes in section 7.

## 1. Imports

In [ ]:
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cupy as cp
from scipy.interpolate import NearestNDInterpolator, RegularGridInterpolator

plt.rc('font', size=10)
plt.rc('axes', titlesize=12)
plt.rc('axes', labelsize=12)
plt.rc('legend', fontsize=9)

# Run from examples/: the CuPy optimalrcs package and the path readers of the examples
REPO = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs
import tis_data
import infretis_data
import seed_runs
FIG = f'{REPO}/examples/figures_by_potential'
sys.path.insert(0, f'{FIG}/_shared/exact_committors')
import lgv2d_tools
# toytis, for the potential contours (lgv2d_tools has the workstation path; use the local checkout if that is missing)
for _p in (lgv2d_tools.TOYTIS, f'{os.path.dirname(os.path.dirname(REPO))}/PyRETIS3/toytis'):
    if os.path.isdir(_p):
        lgv2d_tools.TOYTIS = _p
        break

## 2. Settings

**Run and data**

| Name | Meaning |
|---|---|
| `TIS_DIR` | the infretis simulation folder (`infretis.toml`, `infretis_data.txt`, `load/`) |
| `H5_FILE` | order parameters from an `inft store_txt -order` archive instead of `load/` (`None`: use `load/`) |
| `LAMBDA_A`, `LAMBDA_B` | states A (λ ≤ λ_A) and B (λ ≥ λ_B); `None`: the first and last interface |
| `BURN_IN_FRACTION` | the first part of the rows of `infretis_data.txt` (the order in which the paths ended) is skipped; the initial load paths always are |
| `PATHS_PER_ENSEMBLE`, `THIN` | at most this many paths per ensemble, evenly spaced (`None`: all), after keeping every `THIN`-th |
| `REVERSED` | also use the mirrored pieces (time-reversible dynamics at equilibrium) |
| `VEL_COLUMNS` | the velocity columns of `order.txt` (time = 0, λ = 1, y = 2, x = 3, v_y = 4, v_x = 5): negated in the mirrored pieces |
| `DT` | time per frame, for the velocity check of section 4 |
| `REF_FILE` | the velocity-averaged q̄(y, x) (`y`, `x`, `q`); `None`: no reference |
| `WEIGHTINGS` | the weightings to fit, one fit each: `None` = the run's default, `'staple'` = path_f / path_w, `'rejections'` = path_f, `'none'` = 1 per (path, ensemble) |

**Fit** (the same for every weighting; the history fit uses `HISTORY` and `MAX_ITER_HIST`, the (y, x, v) fit `MAX_ITER_XV`): `SEEDS` (which CV and envelope centre each iteration draws), a warm-up of `WARMUP_ITER` iterations with a wide envelope, then the main iterations, in chunks of `CHUNK` after which the plots are shown and saved. `HISTORY`: the lags d (frames) drawn per iteration for the history fit (0: no lag); the basis is a polynomial in (CV(t − d), CV(t)). A velocity relaxes in 1/γ = 0.5, i.e. 250 frames, so lags of 10–20 frames still measure the velocity, and one frame (a displacement of ~v·dt) is too short for a polynomial basis to resolve. `STOPPED_ENDS` (default `True`): a piece that ends outside A and B was stopped by the path's own end (a turn); the stopped Z_q then uses q at that end for windows that pass it (`set_stopped_traj_end`), and the A/B traps are made long enough that no window is dropped. `NY` is the degree of the polynomial basis, `LAZY_BASIS` evaluates it on the fly (GPU memory).

In [ ]:
TIS_DIR = f'{REPO}/examples/figures_by_potential/_shared/param_lambda/op_loop/loop_z_lgv/production_seed/'
H5_FILE = None  # None: read load/<pnr>/order.txt
LAMBDA_A, LAMBDA_B = 0.05, 0.85     # None: first and last interface
BURN_IN_FRACTION = 0.3
PATHS_PER_ENSEMBLE = 3000           # None: all paths after the burn-in
THIN = 1
REVERSED = True
WEIGHTINGS = ['rejections']         # e.g. [None], ['staple', 'rejections']

VEL_COLUMNS = (4, 5)                # order.txt: time = 0, lambda = 1, y = 2, x = 3, vy = 4, vx = 5
DT = 0.002                          # time per frame
REF_FILE = f'{FIG}/2d_zpot/exact/langevin_g2/qbar_maxwell.npz'   # velocity-averaged q(y, x); None: no reference

SEEDS = [0]
WARMUP_ITER, WARMUP_ENVELOPE = 1000, 1
HISTORY = [0, 10, 20]               # fit A: lags d (frames) drawn each iteration (0: no lag); positions only
MAX_ITER_HIST = 20000               # fit A: main iterations
MAX_ITER_XV = 20000                 # fit B: positions and velocities as CVs, no history
CHUNK = 2500
ENVELOPE, GAMMA = 0.01, 0.05
STOPPED_ENDS = True                 # stopped Z_q: a window passing a turn end of a piece uses q at that end
NY, LAZY_BASIS = 6, True            # basis degree; evaluate the basis on the fly (GPU memory)
PLOTS = ('metrics', 'tis', 'feps', 'obs_pred')          # library plots after every chunk; also possible: 'obs_pred_log'
LABELS = ('order parameter $y$', '$x$')
EDGES = (np.linspace(0.05, 0.85, 81), np.linspace(-0.01, 0.31, 33))   # (y, x) bins for the profiles and maps

RUN_NAME = os.path.basename(TIS_DIR.rstrip('/'))
OUT = f'{FIG}/2d_zpot/langevin_g2/infretis_staple/{RUN_NAME}'
print('figures and results in', OUT)

## 3. The Run

What infretis wrote: the sampling settings, and per ensemble its window (L, M, R), the number of finished paths with a nonzero occupancy, their total occupancy `path_f`, and the share of that occupancy with `path_w` ≠ 1 (where `'staple'` and `'rejections'` differ).

**Weight check** (`infretis_data.weight_consistency`, needs no committor). A path that is valid in two ensembles e and e′ should split its occupancy in the same ratio C_e / C_e′ whatever its type. So the ratio Σ weight_e / Σ weight_e′ over paths of one class (w_e, w_e′) must not depend on the class: relative to class (1, 1), the classes (0.5, 1) and (1, 0.5) should give 1 for the right weighting. Runs with swaps show which one that is. Runs with shooting only have no paths shared between ensembles, and the check is empty.

In [ ]:
settings = infretis_data.read_settings(TIS_DIR)
table = infretis_data.read_infretis_data(os.path.join(TIS_DIR, 'infretis_data.txt'), len(settings['interfaces']))
default_w, why = infretis_data.default_weighting(settings)
print(f"mode {settings['mode']}, infinite swap {settings['infinite_swap']}, staple swap fraction "
      f"{settings['staple_swap_frac']}, {len(table['pnr'])} finished paths")
print(f"interfaces {settings['interfaces']}")
print(f"default weighting: {default_w} ({why})")

f, w = table['path_f'], table['path_w']
rows = []
for e in range(len(settings['interfaces'])):
    on = f[:, e] > 0
    rows.append(dict(ensemble=f'{e:03d}', window=tuple(round(v, 4) for v in infretis_data.ensemble_window(settings, e)),
                     paths=int(on.sum()), occupancy=f[on, e].sum(),
                     share_w_not_1=(f[on, e] * ~np.isclose(w[on, e], 1)).sum() / max(f[on, e].sum(), 1e-12)))
display(pd.DataFrame(rows).set_index('ensemble').round(3))

check = infretis_data.weight_consistency(table)
w_not_1 = ((f > 0) & ~np.isclose(w, 1)).any()
if not w_not_1:
    print("path_w is 1 for every occupied (path, ensemble) pair (no wire fencing, no staple weights): 'staple' (path_f / path_w) "
          "and 'rejections' (path_f) are the same weights for this run, and the weight check below has nothing to compare")
if not check:
    print('weight check: no class of paths with w != 1 that is shared by two ensembles' if not w_not_1 else
          'weight check: no pairs of ensembles with enough shared paths (expected for shooting only)')
else:
    out = []
    for row in check:
        for c in ((0.5, 1), (1, 0.5)):
            if c in row:
                out.append(dict(pair=row['pair'], cls=c, n=row['n'][c],
                                path_f=f"{row[c]['rejections'][0]:.3f} ± {row[c]['rejections'][1]:.3f}",
                                path_f_over_w=f"{row[c]['staple'][0]:.3f} ± {row[c]['staple'][1]:.3f}"))
    display(pd.DataFrame(out))

## 4. From Paths to Frames

`tis_data.prepare_data` reads the run once per weighting in `WEIGHTINGS` (the pieces are the same, only their weights differ). It returns per frame: the CVs, λ, the time within its path (`t_traj`), its piece (`i_traj`), its path (`group_traj`, the infretis path number), the path weight (`path_weights`, mean 1) and the A/B masks. `X_all` has the columns (λ, y, x, v_y, v_x); λ is a copy of y here and is dropped, so the CVs are **(y, x, v_y, v_x)**.

The plots show where the frames are: the weighted number of frames along λ (the path-sampling density, high at the interfaces where pieces start, not the equilibrium distribution), and the piece lengths.

**Velocity check:** within a piece, the recorded velocity must follow the positions: (y(t+1) − y(t)) / DT should correlate with the mean of v_y(t) and v_y(t+1), for every piece, forward or mirrored. A correlation near +1 means the sign flip of the mirrored pieces is right (a missing flip makes it ≈ 0, half of the frames having the wrong sign).

In [ ]:
def prepare(weighting):
    parser = argparse.ArgumentParser()
    tis_data.add_data_arguments(parser)
    argv = [TIS_DIR, '--burn-in-fraction', str(BURN_IN_FRACTION), '--thin', str(THIN)]
    if weighting is not None:
        argv += ['--weights', weighting]
    if H5_FILE is not None:
        argv += ['--h5', H5_FILE]
    if LAMBDA_A is not None:
        argv += ['--lambda-a', str(LAMBDA_A)]
    if LAMBDA_B is not None:
        argv += ['--lambda-b', str(LAMBDA_B)]
    if PATHS_PER_ENSEMBLE is not None:
        argv += ['--paths-per-ensemble', str(PATHS_PER_ENSEMBLE)]
    if REVERSED:
        argv += ['--reversed']
    if VEL_COLUMNS:
        argv += ['--odd-columns'] + [str(c) for c in VEL_COLUMNS]       # negated in the mirrored pieces
    args = parser.parse_args(argv)
    data = tis_data.prepare_data(args)
    assert data.X_all.shape[1] == 5, f'expected the columns (lambda, y, x, vy, vx), got {data.X_all.shape[1]}'
    data.X_all = data.X_all[:, 1:]           # -> (y, x, vy, vx); lambda is a copy of y here
    data.X = data.X_all
    return args.weights, args.lambda_a, args.lambda_b, data     # the weighting actually used


# None = the run's default (section 3); the same weighting twice is fitted once
names = list(dict.fromkeys(w if w is not None else default_w for w in WEIGHTINGS))
datas = {}
for weighting in names:
    name, lambda_a, lambda_b, data = prepare(weighting)
    datas[name] = data
    print(f'--> {name}: {len(data.lam)} frames, {data.i_traj.max() + 1} pieces, '
          f'{len(np.unique(data.group_traj))} paths\n')
data = next(iter(datas.values()))  # the frames are the same for every weighting
free = ~(data.boundary0 | data.boundary1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
e_lam = np.linspace(data.lam.min(), data.lam.max(), 121)
for name, d in datas.items():
    ax1.hist(d.lam, bins=e_lam, weights=d.path_weights, histtype='step', label=name)
for v in settings['interfaces']:
    ax1.axvline(v, color='0.6', lw=0.6)
ax1.set(xlabel=LABELS[0], ylabel='weighted frames', yscale='log')
ax1.legend(title='weighting')
lengths = np.bincount(data.i_traj)
ax2.hist(lengths, bins=np.logspace(0, np.log10(lengths.max() + 1), 60))
ax2.set(xscale='log', yscale='log', xlabel='piece length (frames)', ylabel='pieces')
fig.tight_layout()
plt.show()

# velocity check: recorded velocity against the finite difference of the positions, inside every piece
X = data.X_all
same = data.i_traj[1:] == data.i_traj[:-1]
for k, (pos, vel, lab) in enumerate(((0, 2, 'y'), (1, 3, 'x'))):
    fd = (X[1:, pos] - X[:-1, pos])[same] / DT
    vm = (0.5 * (X[1:, vel] + X[:-1, vel]))[same]
    print(f'v_{lab}: corr(recorded, finite difference) = {np.corrcoef(fd, vm)[0, 1]:+.3f}, '
          f'RMS v = {np.sqrt(np.mean(vm ** 2)):.3f}, RMS finite difference = {np.sqrt(np.mean(fd ** 2)):.3f} '
          f'(thermal: {np.sqrt(0.05):.3f})')

## 5. Reference Committor (optional)

With `REF_FILE`, the velocity-averaged q̄(y, x) of the toytis BBK chain (a 33 × 13 grid; the 8 nodes above 10 kT are NaN and take their nearest neighbour; the frames' positions are clipped to the grid) is the reference for the progress line and the plots: **q̄(y, x) at every frame's position**. As the committor of a frame also depends on its velocity, the comparison is meaningful for bin means (maps, profile), not per frame. `CONTOURS` (grey lines in the maps) are the effective potential of the Z potential (potential plus the constant force along y: `lgv2d_tools.zpot_potential`) in kT above its minimum, every 1 kT up to 12 kT.


In [ ]:
if REF_FILE is None:
    q_exact_of = None
    kT = 0.05
else:
    REF = dict(np.load(REF_FILE))
    assert np.isclose(float(REF['A']), lambda_a) and np.isclose(float(REF['B']), lambda_b), 'the reference has other states A and B'
    kT = float(REF['kT'])
    f_lin = RegularGridInterpolator((REF['y'], REF['x']), REF['q'], bounds_error=False, fill_value=None)
    nodes = np.array([(y, x) for y in REF['y'] for x in REF['x']])
    ok = ~np.isnan(REF['q'].ravel())
    f_near = NearestNDInterpolator(nodes[ok], REF['q'].ravel()[ok])

    def q_exact_of(X):                  # X: the (y, x, ...) columns
        pts = np.c_[np.clip(X[:, 0], REF['y'][0], REF['y'][-1]), np.clip(X[:, 1], REF['x'][0], REF['x'][-1])]
        q = f_lin(pts)
        bad = np.isnan(q)
        q[bad] = f_near(pts[bad])
        q = np.clip(q, 0, 1)
        q[X[:, 0] <= lambda_a], q[X[:, 0] >= lambda_b] = 0, 1
        return q
    print('velocity-averaged reference q(y, x) from', REF_FILE, f'(gamma {float(REF["gamma"])}, kT {kT}, dt {float(REF["dt"])})')

# the potential for the maps, in kT above its minimum (V has shape (len(y), len(x)); the plots use V.T)
_y = np.linspace(lambda_a, lambda_b, 161)
_x = np.linspace(-0.03, 0.33, 73)
_Y, _X = np.meshgrid(_y, _x, indexing='ij')
_V = lgv2d_tools.zpot_potential(_Y, _X) / kT
CONTOURS = (_y, _x, _V - _V.min(), np.arange(0, 12.5, 1.0))

## 6. Fit the Committor

`CommittorNE` with `group_traj` (the pieces of one path count as one unit in the error estimates), the **A/B ends as traps** and the **stopped ends** (`STOPPED_ENDS`), as in `infretis_staple_1d_langevin.ipynb`. Time reversal flips the velocity, so the equilibrium estimator does not apply and the non-equilibrium `CommittorNE` is used.

**Two fits** (cells 6a and 6b), each with every weighting and seed:
- **History, positions only** (`HISTORY`): the CVs are y and x, one drawn per iteration. Each iteration draws a lag d from `HISTORY` (0 = none) and the basis is a polynomial in (CV(t), CV(t − d)). The recorded velocities are not used; the displacement over d frames stands in for them. **Caveat:** the pair (CV(t − d), CV(t)) is not a Markov state, so the committor given the last d frames is not a martingale at longer lags. A small `zqs_z` at lags above the fitted ones cannot be expected; compare the fit with the reference and with fit B instead.
- **Positions and velocities, no history**: the CVs are y, x, v_y and v_x (recorded), one drawn per iteration, with the usual basis in (q, CV). (y, x, v_y, v_x) is a Markov state of the Langevin dynamics, so the committor of the frames is consistent at every lag: here `zqs_z` of about 2 or less is the convergence criterion.

The first cell below defines the plots, the progress line and the piece ends; `run_fit` (cell 6) fits one variant; cells 6a and 6b call it. Plots go to `<OUT>/<variant>/<weighting>/seed<s>/it<N>_<plot>.png`:
- `metrics`, `tis`, `feps`, `obs_pred`: the library plots (the stopped Z_q per lag in `tis`; **convergence is `zqs_z` about 2 or less** with a small `|dx|`);
- `q_profile`: the path-weighted mean q per y bin next to the reference, and fit − reference;
- `q_map`: the mean q per (y, x) bin, the reference q̄ and fit − q̄, on the potential;
- `q_vel`: the mean q per (y, v_y) and (x, v_x) bin: if the velocity matters, q rises with v at fixed position.

The progress line gives the path-weighted RMS(fit − q̄) over the (y, x) bins with 0.05 < q̄ < 0.95.

In [ ]:
def edges_for(d):
    if EDGES is not None:
        return EDGES
    X = d.X_all[~(d.boundary0 | d.boundary1)]
    return tuple(np.linspace(*np.percentile(X[:, k], [0.5, 99.5]), 81 if k == 0 else 33) for k in range(2))


def vel_edges(d):
    fr = ~(d.boundary0 | d.boundary1)
    return tuple(np.linspace(*np.percentile(d.X_all[fr, k], [0.5, 99.5]), 41) for k in (2, 3))


def extras_for(d, qe):
    # q -> figures: the mean q per y bin against the reference, the (y, x) map, the velocity planes
    edges = edges_for(d)
    fr = ~(d.boundary0 | d.boundary1)
    bm = seed_runs.BinnedMean((d.lam,), edges[:1], fr)
    w_mean = bm(d.path_weights)
    mid = (edges[0][1:] + edges[0][:-1]) / 2
    prof_ex = None if qe is None else bm(d.path_weights * qe) / w_mean

    def q_profile(q):
        P = bm(d.path_weights * cp.asnumpy(q.r_traj)) / w_mean
        fig, axes = plt.subplots(1, 1 if qe is None else 2, figsize=(6 if qe is None else 13, 4), squeeze=False)
        axes[0, 0].plot(mid, P, label='fit')
        if qe is not None:
            axes[0, 0].plot(mid, prof_ex, 'k--', label='velocity-averaged reference')
            t = (prof_ex > 0.05) & (prof_ex < 0.95)
            axes[0, 1].plot(mid, P - prof_ex)
            axes[0, 1].axhline(0, color='k', lw=0.8)
            axes[0, 1].set(xlabel=XLABEL, ylabel='fit $-$ reference',
                           title=f'RMS per bin (0.05 < reference < 0.95) {np.sqrt(np.nanmean((P - prof_ex)[t] ** 2)):.4f}')
            axes[0, 1].grid(alpha=0.3)
        axes[0, 0].set(xlabel=XLABEL, ylabel='$q$ (mean per bin)')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)
        fig.tight_layout()
        return fig

    ev = vel_edges(d)
    return {'q_profile': q_profile,
            'q_map': seed_runs.committor_map(
                [seed_runs.plane(d.X_all[:, 0], edges[0], LABELS[0], d.X_all[:, 1], edges[1], LABELS[1],
                                 contours=CONTOURS)], mask=fr, q_ref=qe),
            'q_vel': seed_runs.committor_map(
                [seed_runs.plane(d.X_all[:, 0], edges[0], LABELS[0], d.X_all[:, 2], ev[0], '$v_y$'),
                 seed_runs.plane(d.X_all[:, 1], edges[1], LABELS[1], d.X_all[:, 3], ev[1], '$v_x$')], mask=fr)}


def report_for(d, qe):
    # progress line: path-weighted RMS(fit - reference) over the (y, x) bins with 0.05 < reference < 0.95
    if qe is None:
        return None
    fr = ~(d.boundary0 | d.boundary1)
    bm = seed_runs.BinnedMean((d.X_all[:, 0], d.X_all[:, 1]), edges_for(d), fr)
    w = bm(d.path_weights)
    ref = bm(d.path_weights * qe) / w
    t = (ref > 0.05) & (ref < 0.95)

    def report(q):
        M = bm(d.path_weights * cp.asnumpy(q.r_traj)) / w
        return f'| RMS(fit - reference) per (y, x) bin {np.sqrt(np.nanmean((M - ref)[t] ** 2)):.4f}'
    return report


XLABEL = LABELS[0]

# ---- piece ends: in A or B (trap), or stopped by the path's own end, a turn (stopped end)
# (the frames are the same for every weighting, so this is done once)
b_all = data.boundary0 | data.boundary1
piece_end = np.r_[np.flatnonzero(np.diff(data.i_traj)), len(data.i_traj) - 1]      # last frame of every piece
stop_end = np.zeros(len(b_all), bool)
stop_end[piece_end] = ~b_all[piece_end]
longest = int(np.bincount(data.i_traj - data.i_traj.min()).max())
TRAP_LEN = max(2 ** 16, longest + 2 ** 15 + 1)         # A/B trap: longer than any piece plus the longest Z_q lag (2^15)
print(f'{len(piece_end)} pieces: {stop_end.sum()} end at a turn (stopped), {len(piece_end) - stop_end.sum()} in A or B; '
      f'longest piece {longest} frames, A/B trap {TRAP_LEN}')

PRINT_STEP = min(500, CHUNK, WARMUP_ITER or CHUNK)    # the metrics are recorded every PRINT_STEP iterations (a plot needs a record)
METRICS = ['delta_r2', 'max_z_zq_stopped', 'delta_x']


def show_stage(q, folder, n_done, stage, extras, report):
    """Save and show the plots after a stage (<folder>/it<N>_<plot>.png), and print the progress."""
    seed_runs.save_and_show(q, folder, f'it{n_done}', PLOTS, dict(metrics=METRICS), extras)
    print(f'{n_done} iterations ({stage}) {"" if report is None else report(q)}', flush=True)

In [ ]:
fit_kw = dict(ny=NY, print_step=PRINT_STEP, lazy_basis=LAZY_BASIS, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'))
fits = {}                                      # (variant, weighting) -> final q per seed, reference, fit time, zqs_z history


def run_fit(variant, name, cv_cols, history, max_iter):
    """One fit on the frames of weighting `name`. The CVs are the columns `cv_cols` of X_all (y, x, vy, vx), one drawn
    at random per iteration; `history`: lags d drawn per iteration (None: no history)."""
    d = datas[name]
    qe = q_exact_of(d.X_all) if q_exact_of is not None else None
    X_cv = cp.asarray(d.X_all[:, cv_cols].astype(float))
    extras, report = extras_for(d, qe), report_for(d, qe)
    kw = dict(fit_kw)
    if history:
        kw.update(history_delta_t=list(history), history_type='y(t-d),y(t)')

    def comp_y():
        return X_cv[:, np.random.randint(X_cv.shape[1])]

    t0 = time.time()
    results = {}
    for seed in SEEDS:
        print(f'===== {variant}, weighting {name}, seed {seed} =====', flush=True)
        np.random.seed(seed)
        cp.random.seed(seed)
        folder = f'{OUT}/{variant}/{name}/seed{seed}'
        q = optimalrcs.CommittorNE(boundary0=d.boundary0, boundary1=d.boundary1, i_traj=d.i_traj, t_traj=d.t_traj,
                                   path_weights=d.path_weights, group_traj=d.group_traj)
        q.set_fixed_traj_length_trap(b_all, TRAP_LEN if STOPPED_ENDS else 2 ** 16)
        if STOPPED_ENDS:
            q.set_stopped_traj_end(stop_end)
        n_done = 0
        if WARMUP_ITER:
            optimalrcs.envelope_scale = WARMUP_ENVELOPE
            q.fit_transform(comp_y, gamma=GAMMA, max_iter=WARMUP_ITER, **kw)
            n_done += WARMUP_ITER
            show_stage(q, folder, n_done, f'warm-up, envelope {WARMUP_ENVELOPE}', extras, report)
        optimalrcs.envelope_scale = ENVELOPE
        while n_done < WARMUP_ITER + max_iter:
            n_iter = min(CHUNK, WARMUP_ITER + max_iter - n_done)
            q.fit_transform(comp_y, gamma=GAMMA, max_iter=n_iter, **kw)
            n_done += n_iter
            show_stage(q, folder, n_done, f'main, envelope {ENVELOPE}', extras, report)
        results[seed] = cp.asnumpy(q.r_traj)
        np.save(f'{folder}/q_traj.npy', results[seed].astype(np.float32))
        zqs_z = np.array([float(v) for v in q.metrics_history['max_z_zq_stopped']])
        del q
        cp.get_default_memory_pool().free_all_blocks()
    del X_cv
    fits[(variant, name)] = dict(r=results, q_exact=qe, t_fit=time.time() - t0, zqs_z=zqs_z, weighting=name, variant=variant)

### 6a. Fit A: positions (y, x) with histories

In [ ]:
# Fit A: positions with histories (lag d drawn from HISTORY each iteration)
for name in datas:
    run_fit('history_yx', name, cv_cols=[0, 1], history=HISTORY, max_iter=MAX_ITER_HIST)

### 6b. Fit B: positions and velocities (y, x, v_y, v_x) as CVs, no history

In [ ]:
# Fit B: positions and recorded velocities as CVs, no history
for name in datas:
    run_fit('yx_v', name, cv_cols=[0, 1, 2, 3], history=None, max_iter=MAX_ITER_XV)

## 7. Results

**Table:** per variant, weighting and seed, the data size, the final and lowest `zqs_z` (about 2 or less is consistent; for the history fit see the caveat in section 6) and, with the reference, the path-weighted RMS of fit − q̄ over the (y, x) bins with 0.05 < q̄ < 0.95.

**q along y:** the path-weighted mean q per y bin of every fit (and the reference, dashed), and fit − reference. **q on the (y, x) plane:** fit − q̄ per bin, with q̄ in the first panel (dashed: q = 0.1, 0.5, 0.9; grey: the potential). **q in the velocity planes:** the mean q per (y, v_y) and (x, v_x) bin of every fit; if the velocity matters, q rises with v at fixed position.

In [ ]:
rows, profiles = [], {}
for (variant, name), fit in fits.items():
    d = datas[name]
    fr = ~(d.boundary0 | d.boundary1)
    edges = edges_for(d)
    bm = seed_runs.BinnedMean((d.lam,), edges[:1], fr)
    w_mean = bm(d.path_weights)
    bm2 = seed_runs.BinnedMean((d.X_all[:, 0], d.X_all[:, 1]), edges, fr)
    w2 = bm2(d.path_weights)
    qe = fit['q_exact']
    for s, r in fit['r'].items():
        row = dict(variant=variant, weighting=name, seed=s, paths=len(np.unique(d.group_traj)), frames=len(r),
                   zqs_z_final=fit['zqs_z'][-1], zqs_z_min=fit['zqs_z'][1:].min(), fit_minutes=fit['t_fit'] / 60 / len(fit['r']))
        profiles[(variant, name, s)] = bm(d.path_weights * r) / w_mean
        if qe is not None:
            ref = bm2(d.path_weights * qe) / w2
            t = (ref > 0.05) & (ref < 0.95)
            row['rms_bins_yx'] = np.sqrt(np.nanmean(((bm2(d.path_weights * r) / w2) - ref)[t] ** 2))
            profiles[('reference', name, 0)] = bm(d.path_weights * qe) / w_mean
        rows.append(row)
summary = pd.DataFrame(rows).set_index(['variant', 'weighting', 'seed'])
os.makedirs(OUT, exist_ok=True)
summary.to_csv(f'{OUT}/summary.csv')
display(summary.round(4))

mid = (edges[0][1:] + edges[0][:-1]) / 2
has_ref = any(k[0] == 'reference' for k in profiles)
fig, axes = plt.subplots(2 if has_ref else 1, 1, figsize=(9, 7 if has_ref else 4), sharex=True, squeeze=False)
for (variant, name, s), P in profiles.items():
    if variant == 'reference':
        continue
    axes[0, 0].plot(mid, P, label=f'{variant}, {name}, seed {s}')
    if has_ref:
        axes[1, 0].plot(mid, P - profiles[('reference', name, 0)], label=f'{variant}, {name}, seed {s}')
if has_ref:
    axes[0, 0].plot(mid, profiles[('reference', next(iter(fits))[1], 0)], 'k--', label='velocity-averaged reference')
    axes[1, 0].axhline(0, color='k', lw=0.8)
    axes[1, 0].set(ylabel='fit $-$ reference')
    axes[1, 0].grid(alpha=0.3)
axes[0, 0].set(ylabel='$q$ (mean per bin)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[-1, 0].set(xlabel=LABELS[0])
fig.tight_layout()
fig.savefig(f'{OUT}/profiles.png', dpi=110)
plt.show()

**q on the (y, x) plane.** Mean q per (y, x) bin (bins with at least 20 frames) for every fit and seed, outside A and B; with the reference, fit − q̄ with its RMS over the bins with 0.05 < q̄ < 0.95.

In [ ]:
DIFF_LIM = 0.15                    # colour range of the difference maps
e1, e2 = edges_for(data)
m1, m2 = (e1[1:] + e1[:-1]) / 2, (e2[1:] + e2[:-1]) / 2
first = next(iter(fits))
bm2 = seed_runs.BinnedMean((data.X_all[:, 0], data.X_all[:, 1]), (e1, e2), ~(data.boundary0 | data.boundary1))
w2 = bm2(data.path_weights)
qe0 = fits[first]['q_exact']
ref_map = None if qe0 is None else bm2(data.path_weights * qe0) / w2
panels = [] if ref_map is None else [(ref_map, 'velocity-averaged reference', 'coolwarm', (0, 1))]
for (variant, name), fit in fits.items():
    d = datas[name]
    bmd = seed_runs.BinnedMean((d.X_all[:, 0], d.X_all[:, 1]), (e1, e2), ~(d.boundary0 | d.boundary1))
    wd = bmd(d.path_weights)
    for s, r in fit['r'].items():
        M = bmd(d.path_weights * r) / wd
        if ref_map is None:
            panels.append((M, f'{variant}, {name}, seed {s}', 'coolwarm', (0, 1)))
        else:
            D = M - ref_map
            t = (ref_map > 0.05) & (ref_map < 0.95)
            panels.append((D, f'{variant}, {name}, seed {s} $-$ reference, RMS {np.sqrt(np.nanmean(D[t] ** 2)):.4f}',
                           'PuOr', (-DIFF_LIM, DIFF_LIM)))
n_cols = min(3, len(panels))
n_rows = -(-len(panels) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6.5 * n_cols, 4 * n_rows), squeeze=False)
for ax, (M, title, cmap, (lo, hi)) in zip(axes.ravel(), panels):
    im = ax.pcolormesh(e1, e2, M.T, cmap=cmap, vmin=lo, vmax=hi)
    g1, g2, V, lev = CONTOURS
    ax.contour(g1, g2, V.T, levels=lev, colors='0.45', linewidths=0.5)
    if ref_map is not None:
        ax.contour(m1, m2, np.nan_to_num(ref_map, nan=0.5).T, levels=[0.1, 0.5, 0.9], colors='k',
                   linewidths=0.8, linestyles='--')
    ax.set(title=title, xlabel=LABELS[0], ylabel=LABELS[1], xlim=(e1[0], e1[-1]), ylim=(e2[0], e2[-1]))
    fig.colorbar(im, ax=ax)
for ax in axes.ravel()[len(panels):]:
    ax.axis('off')
fig.tight_layout()
fig.savefig(f'{OUT}/maps.png', dpi=110)
plt.show()

**q in the velocity planes.** Mean q per (y, v_y) and (x, v_x) bin (bins with at least 20 frames) for every fit and seed, outside A and B. If a fit knows the velocity, q rises with v at fixed position; a fit that does not (positions only, no usable history) is constant along v.

In [ ]:
ev_y, ev_x = vel_edges(data)
ey, ex = edges_for(data)
fr0 = ~(data.boundary0 | data.boundary1)
planes = [('y', 0, ey, 2, ev_y, '$v_y$'), ('x', 1, ex, 3, ev_x, '$v_x$')]
fig, axes = plt.subplots(len(fits), 2, figsize=(11, 4 * len(fits)), squeeze=False)
for row, ((variant, name), fit) in enumerate(fits.items()):
    d = datas[name]
    for col, (lab, pc, pe, vc, ve, vlab) in enumerate(planes):
        bmv = seed_runs.BinnedMean((d.X_all[:, pc], d.X_all[:, vc]), (pe, ve), ~(d.boundary0 | d.boundary1))
        wv = bmv(d.path_weights)
        M = bmv(d.path_weights * fit['r'][SEEDS[0]]) / wv
        im = axes[row, col].pcolormesh(pe, ve, M.T, cmap='coolwarm', vmin=0, vmax=1)
        axes[row, col].set(title=f'{variant}, {name}', xlabel=LABELS[pc], ylabel=vlab)
        fig.colorbar(im, ax=axes[row, col])
fig.tight_layout()
fig.savefig(f'{OUT}/q_velocity_planes.png', dpi=110)
plt.show()

## 8. Save

Every fit's q per frame with its frames, in the format of `load_tis_data_cupy.py`'s `committor.npz`, as `<OUT>/<variant>/<weighting>/seed<s>/committor.npz`.

In [ ]:
for (variant, name), fit in fits.items():
    d = datas[name]
    for s, r in fit['r'].items():
        path = f'{OUT}/{variant}/{name}/seed{s}/committor.npz'
        np.savez(path, r=r, lam=d.lam, X_all=d.X_all, i_traj=d.i_traj, t_traj=d.t_traj, group_traj=d.group_traj,
                 path_weights=d.path_weights, boundary0=d.boundary0, boundary1=d.boundary1, weighting=name, variant=variant,
                 tis_dir=TIS_DIR)
        print('saved', path)
